# Accident Model Training (Kaggle)

Trains **1 model**: `ACCIDENT` vs `NORMAL`.

**Model:** EfficientNet-B0 + average over frames (same simple recipe as violence).

## Why this dataset?

The originally listed Hugging Face set
[`ud-smart-city/car-accident-video`](https://huggingface.co/datasets/ud-smart-city/car-accident-video)
is only a **10-clip paid preview**. It has:
- no `ACCIDENT` / `NORMAL` labels
- a `video` column as `VideoDecoder`, not files
- too few clips to train a real classifier

So this notebook trains on a **free CCTV accident dataset** that already has both classes:

https://www.kaggle.com/datasets/ckay16/accident-detection-from-cctv-footage

Typical folders: `train/Accident`, `train/Non Accident` (and test/val).

**How to run:**
1. Open on Kaggle with GPU + Internet
2. Run all cells
3. Download `accident_best.pt`
4. Put it in project `models/accident_best.pt`


In [ ]:
%pip install -q kagglehub opencv-python-headless tqdm scikit-learn


In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import cv2
import kagglehub
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from tqdm.auto import tqdm

SEED = 42
NUM_FRAMES = 8
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 8
LR = 3e-4
NUM_WORKERS = 2
VAL_RATIO = 0.2
MAX_IMAGES_PER_CLASS = None  # e.g. 800 for a faster smoke run

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = OUT_DIR / "accident_best.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

try:
    cv2.setLogLevel(0)
except Exception:
    pass

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
data_root = Path(
    kagglehub.dataset_download("ckay16/accident-detection-from-cctv-footage")
)
print("Dataset path:", data_root)

for pth in sorted(data_root.rglob("*"))[:40]:
    print(pth.relative_to(data_root))


In [ ]:
MEDIA_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".mp4", ".avi", ".mov", ".mkv"}


def normalize_name(name: str) -> str:
    return name.lower().replace("_", "").replace("-", "").replace(" ", "")


# Use ONLY the parent folder of each file.
POSITIVE_FOLDER_NAMES = {"accident", "accidents", "crash", "crashes"}
NEGATIVE_FOLDER_NAMES = {
    "nonaccident",
    "nonaccidents",
    "noaccident",
    "normal",
    "noncrash",
}


def infer_label_from_path(path: Path) -> int | None:
    parent = normalize_name(path.parent.name)
    if parent in NEGATIVE_FOLDER_NAMES:
        return 0  # NORMAL
    if parent in POSITIVE_FOLDER_NAMES:
        return 1  # ACCIDENT
    return None


def collect_media(root: Path):
    items = []
    for path in root.rglob("*"):
        if path.suffix.lower() not in MEDIA_EXTS:
            continue
        label = infer_label_from_path(path)
        if label is None:
            continue
        items.append((path, label))
    return items


all_items = collect_media(data_root)
print("Labeled files found:", len(all_items))
print("ACCIDENT:", sum(1 for _, y in all_items if y == 1))
print("NORMAL:", sum(1 for _, y in all_items if y == 0))

if len(all_items) == 0:
    raise RuntimeError(
        "Could not find Accident / Non Accident folders. Inspect the printed layout above."
    )

if len({y for _, y in all_items}) < 2:
    raise RuntimeError("Only one class found. Need both Accident and Non Accident folders.")

if MAX_IMAGES_PER_CLASS is not None:
    by_class = {0: [], 1: []}
    for item in all_items:
        by_class[item[1]].append(item)
    all_items = []
    for _, rows in by_class.items():
        random.shuffle(rows)
        all_items.extend(rows[:MAX_IMAGES_PER_CLASS])
    print("After capping:", len(all_items))

paths = [p for p, _ in all_items]
labels = [y for _, y in all_items]

train_paths, val_paths, train_y, val_y = train_test_split(
    paths, labels, test_size=VAL_RATIO, random_state=SEED, stratify=labels
)
print("Train:", len(train_paths), "Val:", len(val_paths))


In [ ]:
tv_weights = EfficientNet_B0_Weights.DEFAULT
preprocess = tv_weights.transforms()
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def sample_frames(media_path: Path, num_frames: int = NUM_FRAMES) -> list[np.ndarray]:
    if media_path.suffix.lower() in IMAGE_EXTS:
        bgr = cv2.imread(str(media_path))
        if bgr is None:
            raise RuntimeError(f"Cannot read image: {media_path}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        return [rgb] * num_frames

    cap = cv2.VideoCapture(str(media_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {media_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    frames = []
    if total <= 0:
        while len(frames) < num_frames:
            ok, frame = cap.read()
            if not ok:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    else:
        idxs = np.linspace(0, max(total - 1, 0), num_frames).astype(int)
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            if not ok:
                continue
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        raise RuntimeError(f"No frames from {media_path}")
    while len(frames) < num_frames:
        frames.append(frames[-1])
    return frames[:num_frames]


class ClipDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = list(paths)
        self.labels = list(labels)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = Path(self.paths[idx])
        y = int(self.labels[idx])
        try:
            frames = sample_frames(path)
            tensors = [preprocess(Image.fromarray(f)) for f in frames]
            x = torch.stack(tensors, dim=0)
        except Exception as e:
            print(f"Skip broken file {path}: {e}")
            x = torch.zeros(NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE)
        return x, y


train_loader = DataLoader(
    ClipDataset(train_paths, train_y), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    ClipDataset(val_paths, val_y), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
print("Loaders ready")


In [ ]:
class ClipClassifier(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, t, -1).mean(dim=1)
        return self.head(feats)


model = ClipClassifier().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
print("Model ready")


In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)
        loss_sum += float(loss.item()) * y.size(0)
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(y.size(0))
    return loss_sum / max(total, 1), correct / max(total, 1)


best_acc = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    seen = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for x, y in pbar:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += float(loss.item()) * y.size(0)
        seen += int(y.size(0))
        pbar.set_postfix(loss=running / max(seen, 1))

    train_loss = running / max(seen, 1)
    val_loss, val_acc = evaluate(val_loader)
    print(f"epoch={epoch} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(
            {
                "model_name": "accident",
                "arch": "efficientnet_b0_temporal_mean",
                "num_frames": NUM_FRAMES,
                "img_size": IMG_SIZE,
                "label_map": {"NORMAL": 0, "ACCIDENT": 1},
                "val_acc": best_acc,
                "state_dict": model.state_dict(),
            },
            BEST_PATH,
        )
        print("Saved best ->", BEST_PATH, "acc=", best_acc)

print("Best val accuracy:", best_acc)
print("Download from Kaggle Output:", BEST_PATH)


## After training

1. Download `accident_best.pt`
2. Copy into project: `models/accident_best.pt`
3. In `.env`:
   ```
   ACCIDENT_MODEL_WEIGHTS_PATH=models/accident_best.pt
   ```
4. Wire loading in `src/cctv_ai/inference/accident/model_adapter.py`
